In [66]:
from openai import OpenAI
import rich
import requests

# Configuration
BASE_URL="http://ogxserver-service.llama.svc.cluster.local:8321"
OGX_CONNECTION_URL=BASE_URL + "/v1"

# Policy files
POLICY_FILE_TEXT = "../data/return-policy.txt"
POLICY_FILE_PDF = "../data/sample.pdf"

In [67]:
# Initialize the OpenAI client
client = OpenAI(
    base_url=OGX_CONNECTION_URL,
    api_key="fake",
)

In [73]:
# Upload files 
text_file = client.files.create(
    file=open(POLICY_FILE_TEXT, "rb"),
    purpose="assistants",
)

pdf_file = client.files.create(
    file=open(POLICY_FILE_PDF, "rb"),
    purpose="assistants",
)

print("Text File ID:", text_file.id)
print("PDF File ID:", pdf_file.id)

Text File ID: file-d1278efbcc6e4129b235cab3bb3f5e01
PDF File ID: file-849e802a9bd948329162e97bfb807817


### inline-provider::auto
The following configuration is used in the ConfigMap. The auto provider automatically selects the appropriate processor based on the uploaded file type.
```
file_processors:
  - provider_id: pypdf
    provider_type: inline::pypdf
    config: {}

  - provider_id: markitdown
    provider_type: inline::markitdown
    config: {}

  - provider_id: docling
    provider_type: inline::docling
    config: {}

  - provider_id: auto
    provider_type: inline::auto
    config: {}
```


In [74]:
# inline-provider::auto
# Call file processor endpoint directly for text file

text_response = requests.post(
    f"{BASE_URL}/v1alpha/file-processors/process",
    data={
        "file_id": text_file.id,
        "chunking_strategy": json.dumps({
            "type": "static",
            "static": {
                "max_chunk_size_tokens": 100,
                "chunk_overlap_tokens": 20,
            },
        }),
    },
    headers={
        "Authorization": "Bearer fake",
        "Accept": "application/json",
    },
)

text_response_data = text_response.json()
print("Processor for text file: " + text_response_data["metadata"]["processor"])
print("-----------------")
rich.print(text_response.text)

# Call file processor endpoint directly for pdf file
pdf_response = requests.post(
    f"{BASE_URL}/v1alpha/file-processors/process",
    data={
        "file_id": pdf_file.id,
        "chunking_strategy": json.dumps({
            "type": "static",
            "static": {
                "max_chunk_size_tokens": 100,
                "chunk_overlap_tokens": 20,
            },
        }),
    },
    headers={
        "Authorization": "Bearer fake",
        "Accept": "application/json",
    },
)

pdf_response_data = pdf_response.json()
print("Processor for pdf file: " + pdf_response_data["metadata"]["processor"])
print("-----------------")
rich.print(pdf_response.text)


Processor for text file: text
-----------------


{"chunks":[{"content":"TechMart Return and Refund Policy\nShipping Information:\n- Standard Shipping: 3-5 business 
days (Free on orders over $50)\n- Express Shipping: 1-2 business days ($15.99)\n- Overnight Shipping: Next business
day ($29.99, order before 2 PM EST)\n- Orders are processed within 24 hours on business days\n- Tracking number 
sent via email once shipped\nReturn Time Limits:\n- Standard items can be returned within 30 days of 
delivery","chunk_id":"65230f50-d6fd-e671-c505-b22d2eaa6a5c","metadata":{"document_id":"81a8f60d-24a8-40e2-8758-9c7f
076103e7","filename":"return-policy.txt","file_id":"file-d1278efbcc6e4129b235cab3bb3f5e01","chunk_id":"65230f50-d6f
d-e671-c505-b22d2eaa6a5c","token_count":100,"metadata_token_count":66,"chunk_tokenizer":"tiktoken:cl100k_base"},"ch
unk_metadata":{"chunk_id":"65230f50-d6fd-e671-c505-b22d2eaa6a5c","document_id":"81a8f60d-24a8-40e2-8758-9c7f076103e
7","source":null,"created_timestamp":1782304599,"updated_timestamp":1782304599,"chunk_window":"0-100","chunk_tokeni
zer":"tiktoken:cl100k_base","content_token_count":100,"metadata_token_count":66}},{"content":" email once 
shipped\nReturn Time Limits:\n- Standard items can be returned within 30 days of delivery\n- Electronics must be 
returned within 15 days of delivery\n- Opened software and personalized items cannot be returned\nReturn 
Conditions:\nItems must be in original condition with original packaging intact. All accessories, manuals, and tags
must be included. Items showing signs of use may receive partial refund or be 
rejected.","chunk_id":"7260ae8f-8fdc-dc53-eedf-792a2c934d20","metadata":{"document_id":"81a8f60d-24a8-40e2-8758-9c7
f076103e7","filename":"return-policy.txt","file_id":"file-d1278efbcc6e4129b235cab3bb3f5e01","chunk_id":"7260ae8f-8f
dc-dc53-eedf-792a2c934d20","token_count":82,"metadata_token_count":66,"chunk_tokenizer":"tiktoken:cl100k_base"},"ch
unk_metadata":{"chunk_id":"7260ae8f-8fdc-dc53-eedf-792a2c934d20","document_id":"81a8f60d-24a8-40e2-8758-9c7f076103e
7","source":null,"created_timestamp":1782304599,"updated_timestamp":1782304599,"chunk_window":"80-162","chunk_token
izer":"tiktoken:cl100k_base","content_token_count":82,"metadata_token_count":66}},{"content":" 
rejected.","chunk_id":"2218cf12-fe7b-e628-d569-b9e2380fe034","metadata":{"document_id":"81a8f60d-24a8-40e2-8758-9c7
f076103e7","filename":"return-policy.txt","file_id":"file-d1278efbcc6e4129b235cab3bb3f5e01","chunk_id":"2218cf12-fe
7b-e628-d569-b9e2380fe034","token_count":2,"metadata_token_count":66,"chunk_tokenizer":"tiktoken:cl100k_base"},"chu
nk_metadata":{"chunk_id":"2218cf12-fe7b-e628-d569-b9e2380fe034","document_id":"81a8f60d-24a8-40e2-8758-9c7f076103e7
","source":null,"created_timestamp":1782304599,"updated_timestamp":1782304599,"chunk_window":"160-162","chunk_token
izer":"tiktoken:cl100k_base","content_token_count":2,"metadata_token_count":66}}],"metadata":{"processor":"text","p
rocessing_time_ms":4,"extraction_method":"text","file_size_bytes":754}}

Processor for pdf file: pypdf
-----------------


{"chunks":[{"content":"TechMart Return and Refund Policy\nStandard items can be returned within 30 days of 
delivery.","chunk_id":"7af0b744-296f-11c8-2188-1c94932c5031","metadata":{"document_id":"d17787d7-b1e6-42d2-801c-32f
a39654568","filename":"sample.pdf","page_count":1,"file_id":"file-849e802a9bd948329162e97bfb807817","chunk_id":"7af
0b744-296f-11c8-2188-1c94932c5031","token_count":20,"metadata_token_count":65,"chunk_tokenizer":"tiktoken:cl100k_ba
se"},"chunk_metadata":{"chunk_id":"7af0b744-296f-11c8-2188-1c94932c5031","document_id":"d17787d7-b1e6-42d2-801c-32f
a39654568","source":null,"created_timestamp":1782304599,"updated_timestamp":1782304599,"chunk_window":"0-20","chunk
_tokenizer":"tiktoken:cl100k_base","content_token_count":20,"metadata_token_count":65}}],"metadata":{"processor":"p
ypdf","processing_time_ms":6,"page_count":1,"extraction_method":"pypdf","file_size_bytes":666}}

### Inline File Processor: MarkItDown

Update the file_processors configuration in the ConfigMap to use markitdown:
```
file_processors:
  - provider_id: markitdown
    provider_type: inline::markitdown
    config: {}
```
After updating the ConfigMap, redeploy the configmap and ogxserver.

In [80]:
# Upload files 
text_file = client.files.create(
    file=open(POLICY_FILE_TEXT, "rb"),
    purpose="assistants",
)

pdf_file = client.files.create(
    file=open(POLICY_FILE_PDF, "rb"),
    purpose="assistants",
)

print("Text File ID:", text_file.id)
print("PDF File ID:", pdf_file.id)

Text File ID: file-35f921fbc23a4ebbbd9409898b6d4f9d
PDF File ID: file-13290e5f36824723b80f6c169f1c047a


In [82]:
# Call file processor endpoint directly for text file

response = requests.post(
    f"{BASE_URL}/v1alpha/file-processors/process",
    data={
        "file_id": text_file.id,
        "chunking_strategy": json.dumps({
            "type": "static",
            "static": {
                "max_chunk_size_tokens": 100,
                "chunk_overlap_tokens": 20,
            },
        }),
    },
    headers={
        "Authorization": "Bearer fake",
        "Accept": "application/json",
    },
)

print("Output for text file:")
print("-----------------")
rich.print(text_response.text)

# Call file processor endpoint directly for pdf file
pdf_response = requests.post(
    f"{BASE_URL}/v1alpha/file-processors/process",
    data={
        "file_id": pdf_file.id,
        "chunking_strategy": json.dumps({
            "type": "static",
            "static": {
                "max_chunk_size_tokens": 100,
                "chunk_overlap_tokens": 20,
            },
        }),
    },
    headers={
        "Authorization": "Bearer fake",
        "Accept": "application/json",
    },
)

print("Output for pdf file:")
print("-----------------")
rich.print(pdf_response.text)

Output for text file:
-----------------


{"chunks":[{"content":"TechMart Return and Refund Policy\nShipping Information:\n- Standard Shipping: 3-5 business 
days (Free on orders over $50)\n- Express Shipping: 1-2 business days ($15.99)\n- Overnight Shipping: Next business
day ($29.99, order before 2 PM EST)\n- Orders are processed within 24 hours on business days\n- Tracking number 
sent via email once shipped\nReturn Time Limits:\n- Standard items can be returned within 30 days of 
delivery","chunk_id":"65230f50-d6fd-e671-c505-b22d2eaa6a5c","metadata":{"document_id":"81a8f60d-24a8-40e2-8758-9c7f
076103e7","filename":"return-policy.txt","file_id":"file-d1278efbcc6e4129b235cab3bb3f5e01","chunk_id":"65230f50-d6f
d-e671-c505-b22d2eaa6a5c","token_count":100,"metadata_token_count":66,"chunk_tokenizer":"tiktoken:cl100k_base"},"ch
unk_metadata":{"chunk_id":"65230f50-d6fd-e671-c505-b22d2eaa6a5c","document_id":"81a8f60d-24a8-40e2-8758-9c7f076103e
7","source":null,"created_timestamp":1782304599,"updated_timestamp":1782304599,"chunk_window":"0-100","chunk_tokeni
zer":"tiktoken:cl100k_base","content_token_count":100,"metadata_token_count":66}},{"content":" email once 
shipped\nReturn Time Limits:\n- Standard items can be returned within 30 days of delivery\n- Electronics must be 
returned within 15 days of delivery\n- Opened software and personalized items cannot be returned\nReturn 
Conditions:\nItems must be in original condition with original packaging intact. All accessories, manuals, and tags
must be included. Items showing signs of use may receive partial refund or be 
rejected.","chunk_id":"7260ae8f-8fdc-dc53-eedf-792a2c934d20","metadata":{"document_id":"81a8f60d-24a8-40e2-8758-9c7
f076103e7","filename":"return-policy.txt","file_id":"file-d1278efbcc6e4129b235cab3bb3f5e01","chunk_id":"7260ae8f-8f
dc-dc53-eedf-792a2c934d20","token_count":82,"metadata_token_count":66,"chunk_tokenizer":"tiktoken:cl100k_base"},"ch
unk_metadata":{"chunk_id":"7260ae8f-8fdc-dc53-eedf-792a2c934d20","document_id":"81a8f60d-24a8-40e2-8758-9c7f076103e
7","source":null,"created_timestamp":1782304599,"updated_timestamp":1782304599,"chunk_window":"80-162","chunk_token
izer":"tiktoken:cl100k_base","content_token_count":82,"metadata_token_count":66}},{"content":" 
rejected.","chunk_id":"2218cf12-fe7b-e628-d569-b9e2380fe034","metadata":{"document_id":"81a8f60d-24a8-40e2-8758-9c7
f076103e7","filename":"return-policy.txt","file_id":"file-d1278efbcc6e4129b235cab3bb3f5e01","chunk_id":"2218cf12-fe
7b-e628-d569-b9e2380fe034","token_count":2,"metadata_token_count":66,"chunk_tokenizer":"tiktoken:cl100k_base"},"chu
nk_metadata":{"chunk_id":"2218cf12-fe7b-e628-d569-b9e2380fe034","document_id":"81a8f60d-24a8-40e2-8758-9c7f076103e7
","source":null,"created_timestamp":1782304599,"updated_timestamp":1782304599,"chunk_window":"160-162","chunk_token
izer":"tiktoken:cl100k_base","content_token_count":2,"metadata_token_count":66}}],"metadata":{"processor":"text","p
rocessing_time_ms":4,"extraction_method":"text","file_size_bytes":754}}

Output for pdf file:
-----------------


{"chunks":[{"content":"TechMart Return and Refund Policy\nStandard items can be returned within 30 days of 
delivery.\n\n","chunk_id":"7b4ba292-8bdd-5dd0-51ca-7fe71f27651f","metadata":{"document_id":"c258b8c9-8818-4744-bc30
-8e4f0f3e91b6","filename":"sample.pdf","file_id":"file-13290e5f36824723b80f6c169f1c047a","chunk_id":"7b4ba292-8bdd-
5dd0-51ca-7fe71f27651f","token_count":20,"metadata_token_count":67,"chunk_tokenizer":"tiktoken:cl100k_base"},"chunk
_metadata":{"chunk_id":"7b4ba292-8bdd-5dd0-51ca-7fe71f27651f","document_id":"c258b8c9-8818-4744-bc30-8e4f0f3e91b6",
"source":null,"created_timestamp":1782307711,"updated_timestamp":1782307711,"chunk_window":"0-20","chunk_tokenizer"
:"tiktoken:cl100k_base","content_token_count":20,"metadata_token_count":67}}],"metadata":{"processor":"markitdown",
"processing_time_ms":23,"extraction_method":"markitdown","file_size_bytes":666}}

### Inline File Processor: pypdf

Update the file_processors configuration in the ConfigMap to use pypdf:
```
file_processors:
  - provider_id: pypdf
    provider_type: inline::pypdf
    config: {}
```
After updating the ConfigMap, redeploy the configmap and ogxserver.

In [84]:
# Upload files 
text_file = client.files.create(
    file=open(POLICY_FILE_TEXT, "rb"),
    purpose="assistants",
)

pdf_file = client.files.create(
    file=open(POLICY_FILE_PDF, "rb"),
    purpose="assistants",
)

print("Text File ID:", text_file.id)
print("PDF File ID:", pdf_file.id)

Text File ID: file-f83090bfdefb462e9f5d89d83023f983
PDF File ID: file-9ad60f778f2b4ff7845174456f1d92b6


In [85]:
# Call file processor endpoint directly for text file
response = requests.post(
    f"{BASE_URL}/v1alpha/file-processors/process",
    data={
        "file_id": text_file.id,
        "chunking_strategy": json.dumps({
            "type": "static",
            "static": {
                "max_chunk_size_tokens": 100,
                "chunk_overlap_tokens": 20,
            },
        }),
    },
    headers={
        "Authorization": "Bearer fake",
        "Accept": "application/json",
    },
)

print("Output for text file:")
print("-----------------")
rich.print(text_response.text)

# Call file processor endpoint directly for pdf file
pdf_response = requests.post(
    f"{BASE_URL}/v1alpha/file-processors/process",
    data={
        "file_id": pdf_file.id,
        "chunking_strategy": json.dumps({
            "type": "static",
            "static": {
                "max_chunk_size_tokens": 100,
                "chunk_overlap_tokens": 20,
            },
        }),
    },
    headers={
        "Authorization": "Bearer fake",
        "Accept": "application/json",
    },
)

print("Output for pdf file:")
print("-----------------")
rich.print(pdf_response.text)

Output for text file:
-----------------


{"chunks":[{"content":"TechMart Return and Refund Policy\nShipping Information:\n- Standard Shipping: 3-5 business 
days (Free on orders over $50)\n- Express Shipping: 1-2 business days ($15.99)\n- Overnight Shipping: Next business
day ($29.99, order before 2 PM EST)\n- Orders are processed within 24 hours on business days\n- Tracking number 
sent via email once shipped\nReturn Time Limits:\n- Standard items can be returned within 30 days of 
delivery","chunk_id":"65230f50-d6fd-e671-c505-b22d2eaa6a5c","metadata":{"document_id":"81a8f60d-24a8-40e2-8758-9c7f
076103e7","filename":"return-policy.txt","file_id":"file-d1278efbcc6e4129b235cab3bb3f5e01","chunk_id":"65230f50-d6f
d-e671-c505-b22d2eaa6a5c","token_count":100,"metadata_token_count":66,"chunk_tokenizer":"tiktoken:cl100k_base"},"ch
unk_metadata":{"chunk_id":"65230f50-d6fd-e671-c505-b22d2eaa6a5c","document_id":"81a8f60d-24a8-40e2-8758-9c7f076103e
7","source":null,"created_timestamp":1782304599,"updated_timestamp":1782304599,"chunk_window":"0-100","chunk_tokeni
zer":"tiktoken:cl100k_base","content_token_count":100,"metadata_token_count":66}},{"content":" email once 
shipped\nReturn Time Limits:\n- Standard items can be returned within 30 days of delivery\n- Electronics must be 
returned within 15 days of delivery\n- Opened software and personalized items cannot be returned\nReturn 
Conditions:\nItems must be in original condition with original packaging intact. All accessories, manuals, and tags
must be included. Items showing signs of use may receive partial refund or be 
rejected.","chunk_id":"7260ae8f-8fdc-dc53-eedf-792a2c934d20","metadata":{"document_id":"81a8f60d-24a8-40e2-8758-9c7
f076103e7","filename":"return-policy.txt","file_id":"file-d1278efbcc6e4129b235cab3bb3f5e01","chunk_id":"7260ae8f-8f
dc-dc53-eedf-792a2c934d20","token_count":82,"metadata_token_count":66,"chunk_tokenizer":"tiktoken:cl100k_base"},"ch
unk_metadata":{"chunk_id":"7260ae8f-8fdc-dc53-eedf-792a2c934d20","document_id":"81a8f60d-24a8-40e2-8758-9c7f076103e
7","source":null,"created_timestamp":1782304599,"updated_timestamp":1782304599,"chunk_window":"80-162","chunk_token
izer":"tiktoken:cl100k_base","content_token_count":82,"metadata_token_count":66}},{"content":" 
rejected.","chunk_id":"2218cf12-fe7b-e628-d569-b9e2380fe034","metadata":{"document_id":"81a8f60d-24a8-40e2-8758-9c7
f076103e7","filename":"return-policy.txt","file_id":"file-d1278efbcc6e4129b235cab3bb3f5e01","chunk_id":"2218cf12-fe
7b-e628-d569-b9e2380fe034","token_count":2,"metadata_token_count":66,"chunk_tokenizer":"tiktoken:cl100k_base"},"chu
nk_metadata":{"chunk_id":"2218cf12-fe7b-e628-d569-b9e2380fe034","document_id":"81a8f60d-24a8-40e2-8758-9c7f076103e7
","source":null,"created_timestamp":1782304599,"updated_timestamp":1782304599,"chunk_window":"160-162","chunk_token
izer":"tiktoken:cl100k_base","content_token_count":2,"metadata_token_count":66}}],"metadata":{"processor":"text","p
rocessing_time_ms":4,"extraction_method":"text","file_size_bytes":754}}

Output for pdf file:
-----------------


{"chunks":[{"content":"TechMart Return and Refund Policy\nStandard items can be returned within 30 days of 
delivery.","chunk_id":"7af0b744-296f-11c8-2188-1c94932c5031","metadata":{"document_id":"e942a2a1-76e8-4ccc-92a0-b8b
94e384a35","filename":"sample.pdf","page_count":1,"file_id":"file-9ad60f778f2b4ff7845174456f1d92b6","chunk_id":"7af
0b744-296f-11c8-2188-1c94932c5031","token_count":20,"metadata_token_count":73,"chunk_tokenizer":"tiktoken:cl100k_ba
se"},"chunk_metadata":{"chunk_id":"7af0b744-296f-11c8-2188-1c94932c5031","document_id":"e942a2a1-76e8-4ccc-92a0-b8b
94e384a35","source":null,"created_timestamp":1782307828,"updated_timestamp":1782307828,"chunk_window":"0-20","chunk
_tokenizer":"tiktoken:cl100k_base","content_token_count":20,"metadata_token_count":73}}],"metadata":{"processor":"p
ypdf","processing_time_ms":7,"page_count":1,"extraction_method":"pypdf","file_size_bytes":666}}

### Inline File Processor: docling

Update the file_processors configuration in the ConfigMap to use docling:
```
file_processors:
  - provider_id: docling
    provider_type: inline::docling
    config: {}
```
After updating the ConfigMap, redeploy the configmap and ogxserver.

In [86]:
# Upload files 
text_file = client.files.create(
    file=open(POLICY_FILE_TEXT, "rb"),
    purpose="assistants",
)

pdf_file = client.files.create(
    file=open(POLICY_FILE_PDF, "rb"),
    purpose="assistants",
)

print("Text File ID:", text_file.id)
print("PDF File ID:", pdf_file.id)

Text File ID: file-0542c4b7721145918fecd20ea6584bca
PDF File ID: file-5297d5c393b34a1da532b1028f44c69d


In [87]:
# Call file processor endpoint directly for text file
response = requests.post(
    f"{BASE_URL}/v1alpha/file-processors/process",
    data={
        "file_id": text_file.id,
        "chunking_strategy": json.dumps({
            "type": "static",
            "static": {
                "max_chunk_size_tokens": 100,
                "chunk_overlap_tokens": 20,
            },
        }),
    },
    headers={
        "Authorization": "Bearer fake",
        "Accept": "application/json",
    },
)

print("Output for text file:")
print("-----------------")
rich.print(text_response.text)

# Call file processor endpoint directly for pdf file
pdf_response = requests.post(
    f"{BASE_URL}/v1alpha/file-processors/process",
    data={
        "file_id": pdf_file.id,
        "chunking_strategy": json.dumps({
            "type": "static",
            "static": {
                "max_chunk_size_tokens": 100,
                "chunk_overlap_tokens": 20,
            },
        }),
    },
    headers={
        "Authorization": "Bearer fake",
        "Accept": "application/json",
    },
)

print("Output for pdf file:")
print("-----------------")
rich.print(pdf_response.text)

Output for text file:
-----------------


{"chunks":[{"content":"TechMart Return and Refund Policy\nShipping Information:\n- Standard Shipping: 3-5 business 
days (Free on orders over $50)\n- Express Shipping: 1-2 business days ($15.99)\n- Overnight Shipping: Next business
day ($29.99, order before 2 PM EST)\n- Orders are processed within 24 hours on business days\n- Tracking number 
sent via email once shipped\nReturn Time Limits:\n- Standard items can be returned within 30 days of 
delivery","chunk_id":"65230f50-d6fd-e671-c505-b22d2eaa6a5c","metadata":{"document_id":"81a8f60d-24a8-40e2-8758-9c7f
076103e7","filename":"return-policy.txt","file_id":"file-d1278efbcc6e4129b235cab3bb3f5e01","chunk_id":"65230f50-d6f
d-e671-c505-b22d2eaa6a5c","token_count":100,"metadata_token_count":66,"chunk_tokenizer":"tiktoken:cl100k_base"},"ch
unk_metadata":{"chunk_id":"65230f50-d6fd-e671-c505-b22d2eaa6a5c","document_id":"81a8f60d-24a8-40e2-8758-9c7f076103e
7","source":null,"created_timestamp":1782304599,"updated_timestamp":1782304599,"chunk_window":"0-100","chunk_tokeni
zer":"tiktoken:cl100k_base","content_token_count":100,"metadata_token_count":66}},{"content":" email once 
shipped\nReturn Time Limits:\n- Standard items can be returned within 30 days of delivery\n- Electronics must be 
returned within 15 days of delivery\n- Opened software and personalized items cannot be returned\nReturn 
Conditions:\nItems must be in original condition with original packaging intact. All accessories, manuals, and tags
must be included. Items showing signs of use may receive partial refund or be 
rejected.","chunk_id":"7260ae8f-8fdc-dc53-eedf-792a2c934d20","metadata":{"document_id":"81a8f60d-24a8-40e2-8758-9c7
f076103e7","filename":"return-policy.txt","file_id":"file-d1278efbcc6e4129b235cab3bb3f5e01","chunk_id":"7260ae8f-8f
dc-dc53-eedf-792a2c934d20","token_count":82,"metadata_token_count":66,"chunk_tokenizer":"tiktoken:cl100k_base"},"ch
unk_metadata":{"chunk_id":"7260ae8f-8fdc-dc53-eedf-792a2c934d20","document_id":"81a8f60d-24a8-40e2-8758-9c7f076103e
7","source":null,"created_timestamp":1782304599,"updated_timestamp":1782304599,"chunk_window":"80-162","chunk_token
izer":"tiktoken:cl100k_base","content_token_count":82,"metadata_token_count":66}},{"content":" 
rejected.","chunk_id":"2218cf12-fe7b-e628-d569-b9e2380fe034","metadata":{"document_id":"81a8f60d-24a8-40e2-8758-9c7
f076103e7","filename":"return-policy.txt","file_id":"file-d1278efbcc6e4129b235cab3bb3f5e01","chunk_id":"2218cf12-fe
7b-e628-d569-b9e2380fe034","token_count":2,"metadata_token_count":66,"chunk_tokenizer":"tiktoken:cl100k_base"},"chu
nk_metadata":{"chunk_id":"2218cf12-fe7b-e628-d569-b9e2380fe034","document_id":"81a8f60d-24a8-40e2-8758-9c7f076103e7
","source":null,"created_timestamp":1782304599,"updated_timestamp":1782304599,"chunk_window":"160-162","chunk_token
izer":"tiktoken:cl100k_base","content_token_count":2,"metadata_token_count":66}}],"metadata":{"processor":"text","p
rocessing_time_ms":4,"extraction_method":"text","file_size_bytes":754}}

Output for pdf file:
-----------------


{"chunks":[{"content":"TechMart Return and Refund Policy Standard items can be returned within 30 days of 
delivery.","chunk_id":"455c6a01-371b-7e77-74ad-59d384c53637","metadata":{"document_id":"61ddd4ea-2d5d-47d7-97a5-b68
81f288c45","filename":"sample.pdf","file_id":"file-5297d5c393b34a1da532b1028f44c69d"},"chunk_metadata":{"chunk_id":
"455c6a01-371b-7e77-74ad-59d384c53637","document_id":"61ddd4ea-2d5d-47d7-97a5-b6881f288c45","source":"sample.pdf","
created_timestamp":null,"updated_timestamp":null,"chunk_window":"0","chunk_tokenizer":null,"content_token_count":15
,"metadata_token_count":null}}],"metadata":{"processor":"docling","processing_time_ms":1709,"page_count":1,"extract
ion_method":"docling","file_size_bytes":666}}